# 04 - Cleaning, Validation, and Master Dataset

Notebook ini menggabungkan dataset wisata, kuliner, hotel, dan oleh-oleh menjadi satu `master_dataset.csv`. Proses dibuat sederhana agar mudah dijelaskan pada Bab 3.

In [ ]:
import re
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/processed")
SOURCE_FILES = {
    "wisata": DATA_DIR / "wisata_final.csv",
    "kuliner": DATA_DIR / "kuliner_final.csv",
    "hotel": DATA_DIR / "hotel_final.csv",
    "oleh_oleh": DATA_DIR / "oleh_oleh_final.csv",
}

REQUIRED_COLUMNS = ["name", "category", "rating", "address", "subtypes"]
FINAL_COLUMNS = ["name", "category", "rating", "address", "subtypes", "type", "price_estimate"]
OPTIONAL_PRICE_COLUMNS = ["price", "price_level", "range", "prices"]

SAFE_PRICE_MIN = 10_000
SAFE_PRICE_MAX = 1_000_000

# Estimasi statis, bukan harga real-time dan bukan hasil live API.
TYPE_PRICE_ESTIMATE = {
    "wisata": 50_000,
    "kuliner": 35_000,
    "hotel": 350_000,
    "oleh_oleh": 75_000,
}

CATEGORY_PRICE_RULES = {
    "hotel": 350_000,
    "hostel": 150_000,
    "guest house": 200_000,
    "restaurant": 50_000,
    "cafe": 45_000,
    "coffee": 45_000,
    "tourist": 40_000,
    "museum": 35_000,
    "gift": 75_000,
    "souvenir": 75_000,
    "store": 75_000,
}

print("Source files:")
for dataset_type, path in SOURCE_FILES.items():
    print(f"- {dataset_type}: {path}")

In [ ]:
def normalize_text(value):
    """Membersihkan spasi berlebih dan nilai kosong."""
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip())


def normalize_key(value):
    """Kunci sederhana untuk validasi duplikat."""
    text = normalize_text(value).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_category(value):
    text = normalize_text(value).lower()
    return text if text else "unknown"


def clean_rating(value):
    rating = pd.to_numeric(value, errors="coerce")
    if pd.isna(rating):
        return 0.0
    return float(min(max(rating, 0), 5))


def category_based_price(dataset_type, category, subtypes):
    combined_text = f"{category} {subtypes}".lower()
    for keyword, price in CATEGORY_PRICE_RULES.items():
        if keyword in combined_text:
            return price
    return TYPE_PRICE_ESTIMATE[dataset_type]


def parse_static_price(value):
    """Mengambil angka harga statis dari kolom harga jika tersedia."""
    text = normalize_text(value).lower()
    if not text or text in {"nan", "none", "null", "-"}:
        return None

    # Simbol $ berulang dianggap sebagai price level dari Google Places.
    if re.fullmatch(r"\$+", text):
        level_map = {1: 25_000, 2: 50_000, 3: 100_000, 4: 200_000}
        return level_map.get(len(text), 300_000)

    numbers = []
    for match in re.findall(r"(?:rp|idr)?\s*([0-9][0-9.,]*)", text):
        cleaned = match.replace(".", "").replace(",", "")
        if cleaned.isdigit():
            numbers.append(int(cleaned))

    if not numbers:
        return None

    price = int(sum(numbers) / len(numbers))
    if price < SAFE_PRICE_MIN or price > SAFE_PRICE_MAX:
        return None
    return price


def estimate_price(row, dataset_type):
    """Estimasi aman: pakai harga statis bila valid, jika tidak pakai aturan kategori."""
    for column in OPTIONAL_PRICE_COLUMNS:
        if column in row.index:
            price = parse_static_price(row[column])
            if price is not None:
                return price
    return category_based_price(dataset_type, row["category"], row["subtypes"])

In [ ]:
# Load semua dataset sumber dan tampilkan statistik awal.
raw_frames = {}
for dataset_type, path in SOURCE_FILES.items():
    frame = pd.read_csv(path)
    frame.columns = frame.columns.str.strip().str.lower()
    raw_frames[dataset_type] = frame
    print(f"{dataset_type}: {frame.shape[0]} baris, {frame.shape[1]} kolom")

In [ ]:
clean_frames = []
validation_rows = []

for dataset_type, frame in raw_frames.items():
    missing_columns = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
    if missing_columns:
        raise ValueError(f"{dataset_type} belum memiliki kolom wajib: {missing_columns}")

    working = frame.copy()
    for column in OPTIONAL_PRICE_COLUMNS:
        if column not in working.columns:
            working[column] = pd.NA

    working = working[REQUIRED_COLUMNS + OPTIONAL_PRICE_COLUMNS].copy()

    # Normalisasi teks agar format antar dataset konsisten.
    working["name"] = working["name"].apply(normalize_text)
    working["address"] = working["address"].apply(normalize_text)
    working["category"] = working["category"].apply(normalize_category)
    working["subtypes"] = working["subtypes"].apply(normalize_text).replace("", "unknown")
    working["rating"] = working["rating"].apply(clean_rating)
    working["type"] = dataset_type
    working["name_key"] = working["name"].apply(normalize_key)
    working["address_key"] = working["address"].apply(normalize_key)

    before_drop = len(working)
    working = working[(working["name_key"] != "") & (working["address_key"] != "")].copy()
    critical_rows_removed = before_drop - len(working)

    before_dedup = len(working)
    working = working.drop_duplicates(subset=["name_key", "address_key"], keep="first").copy()
    duplicates_removed = before_dedup - len(working)

    working["price_estimate"] = working.apply(lambda row: estimate_price(row, dataset_type), axis=1)
    working["price_estimate"] = pd.to_numeric(working["price_estimate"], errors="coerce")
    unrealistic_price_count = int(
        working["price_estimate"].isna().sum()
        + ((working["price_estimate"] < SAFE_PRICE_MIN) | (working["price_estimate"] > SAFE_PRICE_MAX)).sum()
    )
    working["price_estimate"] = (
        working["price_estimate"]
        .fillna(TYPE_PRICE_ESTIMATE[dataset_type])
        .clip(SAFE_PRICE_MIN, SAFE_PRICE_MAX)
        .astype("int64")
    )

    clean_frames.append(working[FINAL_COLUMNS])
    validation_rows.append({
        "type": dataset_type,
        "raw_rows": len(frame),
        "clean_rows": len(working),
        "critical_rows_removed": critical_rows_removed,
        "duplicates_removed": duplicates_removed,
        "unknown_category": int((working["category"] == "unknown").sum()),
        "rating_invalid_after_cleaning": int(((working["rating"] < 0) | (working["rating"] > 5)).sum()),
        "unrealistic_price_fixed": unrealistic_price_count,
    })

validation_summary = pd.DataFrame(validation_rows)
validation_summary

In [ ]:
# Gabungkan semua dataset menjadi master dataset.
df_master = pd.concat(clean_frames, ignore_index=True)

df_master["name_key"] = df_master["name"].apply(normalize_key)
df_master["address_key"] = df_master["address"].apply(normalize_key)

global_duplicate_count = int(df_master.duplicated(subset=["name_key", "address_key"]).sum())
df_master = df_master.drop_duplicates(subset=["name_key", "address_key"], keep="first").copy()
df_master = df_master[FINAL_COLUMNS].reset_index(drop=True)

expected_types = {"wisata", "kuliner", "hotel", "oleh_oleh"}
actual_types = set(df_master["type"].unique())
if actual_types != expected_types:
    raise AssertionError(f"Tipe dataset tidak lengkap. Expected {expected_types}, got {actual_types}")

if df_master[FINAL_COLUMNS].isna().sum().sum() != 0:
    raise AssertionError("Masih ada nilai null pada kolom final.")

if df_master.duplicated(subset=["name", "address"]).any():
    raise AssertionError("Masih ada duplikat berdasarkan name + address.")

if ((df_master["rating"] < 0) | (df_master["rating"] > 5)).any():
    raise AssertionError("Rating harus berada pada rentang 0 sampai 5.")

if ((df_master["price_estimate"] < SAFE_PRICE_MIN) | (df_master["price_estimate"] > SAFE_PRICE_MAX)).any():
    raise AssertionError("price_estimate berada di luar rentang aman.")

print("Master dataset shape:", df_master.shape)
print("Global duplicates removed:", global_duplicate_count)
print("Null count:", int(df_master.isna().sum().sum()))
print("Type counts:")
print(df_master["type"].value_counts())

In [ ]:
# Ringkasan kategori dan harga untuk kebutuhan dokumentasi.
print("Category counts (top 10):")
print(df_master["category"].value_counts().head(10))

print("Price estimate summary:")
print(df_master["price_estimate"].describe().astype("int64"))

In [ ]:
# Contoh output master dataset.
df_master.head(10)

In [ ]:
# Simpan master dataset final.
output_path = DATA_DIR / "master_dataset.csv"
df_master.to_csv(output_path, index=False)
print(f"Saved: {output_path}")